# Ownership Chain Reconstruction 

**Input:** `data/merged/all_conflicts.csv`

In [1]:
import pandas as pd
import requests
import time
import re
from collections import defaultdict, deque
import networkx as nx


In [2]:
# Upload merged dataset and prepare unique manufacturer

INPUT_PATH = 'data/merged/all_conflicts.csv'

df = pd.read_csv(INPUT_PATH)

print(f'Dataset loaded: {len(df)} rows, {df["conflict"].nunique()} conflicts')
print(f'Conflicts: {sorted(df["conflict"].unique())}')

# label cleaning
qid_like = re.compile(r'^Q\d+$')
def is_placeholder_label(x):
    if pd.isna(x): return True
    return bool(qid_like.match(str(x).strip()))

for col in ['participantLabel','weaponLabel','manufacturerLabel',
            'typeLabel','manufacturerCountryLabel','originLabel']:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

# filter rows with manufacturer
df_m = df[df['manufacturer'].replace('', pd.NA).notna()].copy()
df_m = df_m[~df_m['manufacturerLabel'].apply(is_placeholder_label)]
df_m = df_m.drop_duplicates(subset=['conflict','weapon','manufacturer'])

# unique manufacturer cross-conflict
manufacturers = (
    df_m.groupby(['manufacturer','manufacturerLabel',
                  'manufacturerCountry','manufacturerCountryLabel'])
    .agg(conflicts=('conflict', lambda x: '|'.join(sorted(x.unique()))))
    .reset_index()
    .sort_values('manufacturerLabel')
    .reset_index(drop=True)
)

manufacturers.to_csv('manufacturers_unique.csv', index=False)
df_m.to_csv('weapon_manufacturer_clean.csv', index=False)

print(f'\nClean rows with manufacturer: {len(df_m)}')
print(f'Unique manufacturer cross-conflict: {len(manufacturers)}')
print(f'\nManufacturer by number of conflicts covered:')
print(
    manufacturers['conflicts']
    .apply(lambda x: len(x.split('|')))
    .value_counts()
    .sort_index(ascending=False)
    .rename_axis('n_conflicts')
    .reset_index(name='n_manufacturer')
    .to_string(index=False)
)
print(f'\nTop 15 manufacturer by conflicts covered:')
top = manufacturers.copy()
top['n_conflicts'] = top['conflicts'].apply(lambda x: len(x.split('|')))
print(top.nlargest(15, 'n_conflicts')[['manufacturerLabel','n_conflicts','conflicts']].to_string(index=False))


Dataset loaded: 835 rows, 9 conflicts
Conflicts: ['afghanistan', 'gaza', 'iran_iraq', 'iraq', 'kosovo_1999', 'lebanon_2006', 'nagorno_karabakh', 'syria', 'ukraine']

Clean rows with manufacturer: 677
Unique manufacturer cross-conflict: 196

Manufacturer by number of conflicts covered:
 n_conflicts  n_manufacturer
           6               5
           5               6
           4              25
           3              15
           2              72
           1              73

Top 15 manufacturer by conflicts covered:
               manufacturerLabel  n_conflicts                                              conflicts
                    Aérospatiale            6   afghanistan|gaza|iraq|kosovo_1999|lebanon_2006|syria
                          Boeing            6   afghanistan|gaza|iraq|kosovo_1999|lebanon_2006|syria
 Israel Aerospace Industries Ltd            6   afghanistan|gaza|iraq|kosovo_1999|lebanon_2006|syria
    Northrop Grumman Corporation            6 afghanistan|gaza|i

In [3]:
# Extract ownership edges from Wikidata (iterative expansion)

INPUT     = 'manufacturers_unique.csv' 
OUT_EDGES = 'ownership_edges.csv'
OUT_NODES = 'ownership_nodes.csv'

SPARQL_URL = 'https://query.wikidata.org/sparql'
HEADERS = {
    'User-Agent': 'KEN4256-OwnershipBot/1.0 (student project)',
    'Accept':     'application/sparql-results+json'
}

MAX_DEPTH  = 5
BATCH_SIZE = 80   


TERMINAL_P31_QIDS = {
    'Q3624078',  # sovereign state
    'Q6256',     # country
    'Q7275',     # state
    'Q5',        # human (persona fisica)
}

qid_re = re.compile(r'(Q\d+)')
def to_qid(x):
    if pd.isna(x): return None
    m = qid_re.search(str(x))
    return m.group(1) if m else None

def to_uri(qid):
    return f'http://www.wikidata.org/entity/{qid}'

def chunk(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

def sparql_query(q, retries=3, sleep_s=2.0):
    for attempt in range(retries):
        try:
            r = requests.get(SPARQL_URL, params={'query': q}, headers=HEADERS, timeout=60)
            if r.status_code == 200:
                return r.json()
            time.sleep(sleep_s * (2 ** attempt))
        except Exception as e:
            print(f'    attempt {attempt+1} failed: {e}')
            time.sleep(sleep_s * (2 ** attempt))
    raise RuntimeError(f'SPARQL failed after {retries} retries')

OWNERSHIP_QUERY = '''
PREFIX wd:  <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
SELECT DISTINCT ?node ?owner ?ownerCountry ?rel WHERE {{
  VALUES ?node {{ {values} }}
  {{ ?node wdt:P749 ?owner . BIND("P749_parent_org" AS ?rel) }}
  UNION
  {{ ?node wdt:P127 ?owner . BIND("P127_owned_by" AS ?rel) }}
  UNION
  {{ ?owner wdt:P355 ?node . BIND("P355_subsidiary_of_owner" AS ?rel) }}
  UNION
  {{ ?node wdt:P361 ?owner . BIND("P361_part_of" AS ?rel) }}
  UNION
  {{
    FILTER NOT EXISTS {{ ?node wdt:P749 [] }}
    FILTER NOT EXISTS {{ ?node wdt:P127 [] }}
    FILTER NOT EXISTS {{ [] wdt:P355 ?node }}
    ?node wdt:P159 ?hq .
    ?hq wdt:P17 ?owner .
    BIND("P159_hq_country_fallback" AS ?rel)
  }}
  OPTIONAL {{ ?owner wdt:P17 ?ownerCountry . }}
}}
'''


P31_QUERY = '''
PREFIX wd:  <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
SELECT DISTINCT ?entity ?p31 WHERE {{
  VALUES ?entity {{ {values} }}
  OPTIONAL {{ ?entity wdt:P31 ?p31 . }}
}}
'''


df_manu   = pd.read_csv(INPUT)
seed_qids = sorted({to_qid(x) for x in df_manu['manufacturer'].tolist() if to_qid(x)})
print(f'Seed manufacturer: {len(seed_qids)}')


all_edges  = []          
visited    = set()      
p31_cache  = {}         

frontier   = set(seed_qids)
depth      = 0

print(f'\nStart iterative expansion (MAX_DEPTH={MAX_DEPTH})...')
print('─' * 60)

while frontier and depth < MAX_DEPTH:
    print(f'\nIteration {depth + 1} — nodes to explore: {len(frontier)}')

    iter_edges = []
    frontier_list = sorted(frontier)

    # Step A: fetch ownership edges for all nodes in frontier
    for batch_i, qids in enumerate(chunk(frontier_list, BATCH_SIZE), start=1):
        values = ' '.join(f'wd:{q}' for q in qids)
        data   = sparql_query(OWNERSHIP_QUERY.format(values=values))
        rows   = data['results']['bindings']
        for row in rows:
            iter_edges.append({
                'manufacturer': row['node']['value'],
                'owner':        row['owner']['value'],
                'ownerCountry': row.get('ownerCountry', {}).get('value', ''),
                'rel':          row['rel']['value'],
                'depth':        depth
            })
        print(f'  batch {batch_i}: +{len(rows)} edges')
        time.sleep(1.0)

    all_edges.extend(iter_edges)
    visited.update(frontier)

    # Step B: discover new owners
    new_owner_qids = {
        to_qid(e['owner'])
        for e in iter_edges
        if to_qid(e['owner'])
    } - visited

    print(f'  new owners discovered: {len(new_owner_qids)}')

    if not new_owner_qids:
        print('  → frontier empty, natural convergence')
        break

    # Step C: classify new owners via P31
    # remove semantic terminals from frontier
    new_owner_list = sorted(new_owner_qids)
    for batch_i, qids in enumerate(chunk(new_owner_list, BATCH_SIZE), start=1):
        values = ' '.join(f'wd:{q}' for q in qids)
        data   = sparql_query(P31_QUERY.format(values=values))
        for row in data['results']['bindings']:
            eid  = to_qid(row['entity']['value'])
            p31  = to_qid(row.get('p31', {}).get('value', ''))
            if eid:
                p31_cache.setdefault(eid, set())
                if p31: p31_cache[eid].add(p31)
        time.sleep(0.5)

    # rimuovi terminali semantici dalla frontier
    terminals_found = {
        q for q in new_owner_qids
        if p31_cache.get(q, set()) & TERMINAL_P31_QIDS
    }

    if terminals_found:
        print(f'  terminali semantici rimossi dalla frontier: {len(terminals_found)}')

    frontier = new_owner_qids - terminals_found
    depth   += 1

    print(f'  frontier prossima iterazione: {len(frontier)} nodi')

if depth >= MAX_DEPTH:
    print(f'\n⚠ Stop per MAX_DEPTH={MAX_DEPTH} raggiunta')
else:
    print(f'\n✓ Convergenza raggiunta in {depth} iterazioni')

# salva edges
edges_df = pd.DataFrame(all_edges).drop_duplicates(subset=['manufacturer','owner','rel'])
edges_df.to_csv(OUT_EDGES, index=False)
print(f'\nEdge totali salvati: {len(edges_df)} → {OUT_EDGES}')
print(f'Distribuzione per rel:\n{edges_df["rel"].value_counts()}')

# nodes collection for labeling
Q_LABEL = '''
PREFIX wd:  <http://www.wikidata.org/entity/>
PREFIX bd:  <http://www.bigdata.com/rdf#>
PREFIX wikibase: <http://wikiba.se/ontology#>
SELECT DISTINCT ?entity ?entityLabel WHERE {{
  VALUES ?entity {{ {values} }}
  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
}}
'''

Q_P31 = '''
PREFIX wd:  <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX bd:  <http://www.bigdata.com/rdf#>
PREFIX wikibase: <http://wikiba.se/ontology#>
SELECT DISTINCT ?entity ?p31 ?p31Label WHERE {{
  VALUES ?entity {{ {values} }}
  ?entity wdt:P31 ?p31 .
  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
}}
'''

Q_P17 = '''
PREFIX wd:  <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX bd:  <http://www.bigdata.com/rdf#>
PREFIX wikibase: <http://wikiba.se/ontology#>
SELECT DISTINCT ?entity ?country ?countryLabel WHERE {{
  VALUES ?entity {{ {values} }}
  ?entity wdt:P17 ?country .
  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
}}
'''

all_entities = set()
all_entities.update(edges_df['manufacturer'].dropna())
all_entities.update(edges_df['owner'].dropna())
all_entities.update(edges_df['ownerCountry'].dropna())
all_entities.update([to_uri(q) for q in seed_qids])
all_entities = sorted(x for x in all_entities if x and str(x) != 'nan')
print(f'\nEntities to label: {len(all_entities)}')

entity_labels = {}
p31_rows      = []
p17_rows      = []

LABEL_BATCH = 100   
P31_BATCH   = 60    
P17_BATCH   = 100

print('  Step 1/3: entity labels...')
for batch_i, ents in enumerate(chunk(all_entities, LABEL_BATCH), start=1):
    values = ' '.join(f'<{e}>' for e in ents)
    data   = sparql_query(Q_LABEL.format(values=values))
    for row in data['results']['bindings']:
        e   = row['entity']['value']
        lab = row.get('entityLabel', {}).get('value', '')
        entity_labels[e] = lab
    print(f'    batch {batch_i}: {len(data["results"]["bindings"])} labels')
    time.sleep(1.5)

print('  Step 2/3: P31 (instance of)...')
for batch_i, ents in enumerate(chunk(all_entities, P31_BATCH), start=1):
    values = ' '.join(f'<{e}>' for e in ents)
    data   = sparql_query(Q_P31.format(values=values))
    for row in data['results']['bindings']:
        p31_rows.append({
            'entity':   row['entity']['value'],
            'p31':      row['p31']['value'],
            'p31Label': row.get('p31Label', {}).get('value', ''),
        })
    print(f'    batch {batch_i}: {len(data["results"]["bindings"])} p31 rows')
    time.sleep(1.5)

print('  Step 3/3: P17 (country)...')
for batch_i, ents in enumerate(chunk(all_entities, P17_BATCH), start=1):
    values = ' '.join(f'<{e}>' for e in ents)
    data   = sparql_query(Q_P17.format(values=values))
    for row in data['results']['bindings']:
        p17_rows.append({
            'entity':       row['entity']['value'],
            'country':      row['country']['value'],
            'countryLabel': row.get('countryLabel', {}).get('value', ''),
        })
    print(f'    batch {batch_i}: {len(data["results"]["bindings"])} p17 rows')
    time.sleep(1.5)

# assemble nodes_df by joining the 3 results
base_rows = [{'entity': e, 'entityLabel': entity_labels.get(e, '')} for e in all_entities]
base_df   = pd.DataFrame(base_rows)

p31_df = pd.DataFrame(p31_rows) if p31_rows else pd.DataFrame(columns=['entity','p31','p31Label'])
p17_df = pd.DataFrame(p17_rows) if p17_rows else pd.DataFrame(columns=['entity','country','countryLabel'])

# merge
nodes_df = (
    base_df
    .merge(p31_df, on='entity', how='left')
    .merge(p17_df, on='entity', how='left')
    .fillna('')
)

nodes_df = nodes_df.drop_duplicates(subset=['entity','p31'])
nodes_df.to_csv(OUT_NODES, index=False)
print(f'Nodes saved: {len(nodes_df)} → {OUT_NODES}')


Seed manufacturer: 196

Start iterative expansion (MAX_DEPTH=5)...
────────────────────────────────────────────────────────────

Iteration 1 — nodes to explore: 196
  batch 1: +150 edges
  batch 2: +179 edges
  batch 3: +77 edges
  new owners discovered: 146
  terminali semantici rimossi dalla frontier: 23
  frontier prossima iterazione: 123 nodi

Iteration 2 — nodes to explore: 123
  batch 1: +156 edges
  batch 2: +148 edges
  new owners discovered: 86
  terminali semantici rimossi dalla frontier: 14
  frontier prossima iterazione: 72 nodi

Iteration 3 — nodes to explore: 72
  batch 1: +211 edges
  new owners discovered: 64
  terminali semantici rimossi dalla frontier: 21
  frontier prossima iterazione: 43 nodi

Iteration 4 — nodes to explore: 43
  batch 1: +61 edges
  new owners discovered: 37
  terminali semantici rimossi dalla frontier: 17
  frontier prossima iterazione: 20 nodi

Iteration 5 — nodes to explore: 20
  batch 1: +24 edges
  new owners discovered: 13
  terminali semanti

In [4]:
# BFS ultimate owners 

EDGES     = 'ownership_edges.csv'
NODES     = 'ownership_nodes.csv'
MANU      = 'manufacturers_unique.csv'
OUT_ULT   = 'ultimate_owners.csv'
OUT_PATHS = 'ultimate_paths.csv'

edges = pd.read_csv(EDGES)
nodes = pd.read_csv(NODES)
manu  = pd.read_csv(MANU)

# ── lookups
for col in ['entity','entityLabel','p31Label','p31','country','countryLabel']:
    if col in nodes.columns:
        nodes[col] = nodes[col].fillna('').astype(str)

label_by     = nodes.groupby('entity')['entityLabel'].first().to_dict()
p31labels_by = (nodes.groupby('entity')['p31Label']
    .apply(lambda s: {x.strip().lower() for x in s if str(x).strip()}).to_dict())
p31uris_by   = (nodes.groupby('entity')['p31']
    .apply(lambda s: {x.strip() for x in s if str(x).strip()}).to_dict())
country_by   = nodes.groupby('entity')['countryLabel'].first().to_dict()

def label(e):
    if not e or (isinstance(e, float) and pd.isna(e)): return ''
    s = str(e)
    lab = str(label_by.get(s, '')).strip()
    return lab if lab else (s.rsplit('/', 1)[-1] if '/' in s else s)

def p31labels(e): return p31labels_by.get(str(e), set()) if e else set()
def p31uris(e):   return p31uris_by.get(str(e), set()) if e else set()
def get_country(e): return country_by.get(str(e), '') if e else ''

# build parent map
parents = defaultdict(set)
for _, r in edges.iterrows():
    if r.get('rel', '') == 'P361_part_of':
        continue
    m, o = str(r.get('manufacturer','')), str(r.get('owner',''))
    if m and o and m != 'nan' and o != 'nan':
        parents[m].add(o)

print(f'Edge P361 esclusi: {(edges["rel"] == "P361_part_of").sum()}')
print(f'Edge usati per BFS: {sum(len(v) for v in parents.values())}')

# classifiers
INVESTOR_KW = {'asset manager','investment company','investment management',
               'mutual fund','private equity firm','sovereign wealth fund'}
BANK_KW     = {'bank','investment bank','credit institution','financial institution'}

# QID terminals for all_links: sovereign states, countries, state organizations
STATE_QIDS = {
    'http://www.wikidata.org/entity/Q3624078',  # sovereign state
    'http://www.wikidata.org/entity/Q6256',     # country
    'http://www.wikidata.org/entity/Q7275',     # state
}
SOE_QIDS = {
    'http://www.wikidata.org/entity/Q270791',   # state-owned enterprise
    'http://www.wikidata.org/entity/Q219577',   # holding company
    'http://www.wikidata.org/entity/Q327333',   # government agency
    'http://www.wikidata.org/entity/Q192350',   # ministry
}

# Whitelist conglomerates for controlish mode
DEFENSE_CONGLOMERATES = {
    # ── Russia
    'http://www.wikidata.org/entity/Q1840188',   # Rostec
    'http://www.wikidata.org/entity/Q1331507',   # Almaz-Antey
    'http://www.wikidata.org/entity/Q980831',    # Oboronprom
    'http://www.wikidata.org/entity/Q1136226',   # United Aircraft Corporation
    'http://www.wikidata.org/entity/Q1781702',   # Sistema
    'http://www.wikidata.org/entity/Q1284333',   # OMZ
    # ── USA
    'http://www.wikidata.org/entity/Q66',        # Boeing
    'http://www.wikidata.org/entity/Q7240',      # Lockheed Martin
    'http://www.wikidata.org/entity/Q329953',    # Northrop Grumman Corporation
    'http://www.wikidata.org/entity/Q502940',    # General Dynamics
    'http://www.wikidata.org/entity/Q156644',    # Raytheon
    'http://www.wikidata.org/entity/Q54173',     # General Electric
    'http://www.wikidata.org/entity/Q81965',     # General Motors
    'http://www.wikidata.org/entity/Q459965',    # Caterpillar Inc.
    'http://www.wikidata.org/entity/Q30291091',  # L3Harris Technologies
    'http://www.wikidata.org/entity/Q2140315',   # Textron
    # ── Europa
    'http://www.wikidata.org/entity/Q161544',    # Rheinmetall AG
    'http://www.wikidata.org/entity/Q739416',    # BAE Systems
    'http://www.wikidata.org/entity/Q121010578', # Finmeccanica
    'http://www.wikidata.org/entity/Q243278',    # Rolls-Royce
    'http://www.wikidata.org/entity/Q1161666',   # Thales Group
    # ── Israele
    'http://www.wikidata.org/entity/Q1325369',   # Elbit Systems Ltd.
    'http://www.wikidata.org/entity/Q876017',    # Israel Aerospace Industries Ltd
    'http://www.wikidata.org/entity/Q154610',    # Rafael Advanced Defense Systems
    # ── Cina
    'http://www.wikidata.org/entity/Q2667719',   # Aviation Industry Corporation of China, Ltd.
    # ── Altro
    'http://www.wikidata.org/entity/Q158888',    # Yamaha Motor Company
    'http://www.wikidata.org/entity/Q56466264',  # OMPO Holding
    'http://www.wikidata.org/entity/Q3739421',   # Ukrainian Defense Industry
}

def _kw(e, kws): return any(k in ' '.join(sorted(p31labels(e))) for k in kws)

def is_investor(e): return _kw(e, INVESTOR_KW) or label(e) in {
    'BlackRock','The Vanguard Group','Fidelity Investments',
    'Capital Group Companies','State Street Corporation',
    'Longview Asset Management','Cerberus Capital Management',
    'ValueAct Capital','Capital Research Global Investors',
    'Geode Capital Management','Goldman Sachs Capital Partners',
    'Newport Trust','Newport Trust Company',
    'Franklin Templeton Investments','MFS Investment Management',
    'Harris Associates','T. Rowe Price','Wellington Management Company',
    'Sun Life Financial'}
def is_bank(e): return _kw(e, BANK_KW) or label(e) in {
    'Société Générale'}  # P31=organization, causa Rheinmetall→France
def is_human(e):  return 'human' in ' '.join(sorted(p31labels(e)))

def is_state_like(e):
    if p31uris(e) & STATE_QIDS: return True
    return any(k in ' '.join(sorted(p31labels(e)))
               for k in {'government','federal government','sovereign state','country'})

def is_soe(e):
    if p31uris(e) & SOE_QIDS: return True
    return any(k in ' '.join(sorted(p31labels(e)))
               for k in {'state-owned','government-owned','ministry','government agency'})

def is_conglomerate(e):
    return str(e) in DEFENSE_CONGLOMERATES


def ultimate_owners(start, max_depth=8, control_mode=False):
    terminals, example_paths = set(), []
    q = deque([(start, [start])])
    seen = {start}

    while q:
        node, path = q.popleft()
        depth = len(path) - 1

        if depth >= max_depth:
            terminals.add(node)
            if len(example_paths) < 5: example_paths.append(path)
            continue

        nexts = parents.get(node, set())
        if not nexts:
            terminals.add(node)
            if len(example_paths) < 5: example_paths.append(path)
            continue

        for nxt in nexts:
            if nxt in seen: continue
            if control_mode:
                if is_investor(nxt) or is_bank(nxt) or is_human(nxt):
                    terminals.add(node)
                    if len(example_paths) < 5: example_paths.append(path)
                    continue
                if is_conglomerate(nxt):
                    terminals.add(nxt)
                    if len(example_paths) < 5: example_paths.append(path + [nxt])
                    continue
                if is_state_like(nxt) or is_soe(nxt):
                    terminals.add(nxt)
                    if len(example_paths) < 5: example_paths.append(path + [nxt])
                    continue
            else:  
                if is_investor(nxt) or is_bank(nxt):
                    terminals.add(nxt)
                    if len(example_paths) < 5: example_paths.append(path + [nxt])
                    continue
            seen.add(nxt)
            q.append((nxt, path + [nxt]))

    return terminals, example_paths


conflict_for_manu = manu.set_index('manufacturer')['conflicts'].to_dict()


manu_uris = []
for x in manu['manufacturer'].dropna():
    s = str(x).strip()
    if s: manu_uris.append(s if s.startswith('http') else f'http://www.wikidata.org/entity/{s}')
manu_uris = sorted(set(manu_uris))

rows_ult, rows_paths = [], []

for m in manu_uris:
    m_conflict = conflict_for_manu.get(m, '')
    for mode, ctrl in [('all_links', False), ('controlish', True)]:
        terms, paths = ultimate_owners(m, control_mode=ctrl)

        for t in terms:
            rows_ult.append({
                'conflict':             m_conflict,
                'manufacturer':         m,
                'manufacturerLabel':    label(m),
                'ultimateOwner':        t,
                'ultimateOwnerLabel':   label(t),
                'ultimateOwnerCountry': get_country(t),
                'mode':                 mode
            })

        for p in paths:
            rows_paths.append({
                'conflict':          m_conflict,
                'manufacturer':      m,
                'manufacturerLabel': label(m),
                'mode':              mode,
                'depth':             len(p) - 1,
                'path':              ' -> '.join(label(x) for x in p)
            })

ult_df   = pd.DataFrame(rows_ult).drop_duplicates()
paths_df = pd.DataFrame(rows_paths).drop_duplicates()

ult_df.to_csv(OUT_ULT, index=False)
paths_df.to_csv(OUT_PATHS, index=False)

print(f'\nUltimate owners: {len(ult_df)} rows -> {OUT_ULT}')
print(f'Paths: {len(paths_df)} rows -> {OUT_PATHS}')
print('\nDistribution by mode:')
print(ult_df.groupby('mode')[['manufacturer','ultimateOwner']].nunique())




Edge P361 esclusi: 164
Edge usati per BFS: 524

Ultimate owners: 649 rows -> ultimate_owners.csv
Paths: 607 rows -> ultimate_paths.csv

Distribution by mode:
            manufacturer  ultimateOwner
mode                                   
all_links            196            108
controlish           196            102


In [5]:
# sanity check base 

ult = pd.read_csv('ultimate_owners.csv')

print(f'Total rows: {len(ult)}')
print(f'Total manufacturers: {ult["manufacturer"].nunique()}')
print(f'\nDistribution by mode:')
print(ult.groupby('mode')[['manufacturer','ultimateOwner']].nunique())

for mode in ult['mode'].unique():
    sub  = ult[ult['mode'] == mode].copy()
    sub2 = sub[sub['manufacturer'] != sub['ultimateOwner']].copy()
    with_owner = set(sub2['manufacturer'])
    no_owner   = set(sub['manufacturer']) - with_owner

    print(f'\n== {mode} ==')
    print(f'  total manufacturers:          {sub["manufacturer"].nunique()}')
    print(f'  with at least 1 non-self owner:  {len(with_owner)}')
    print(f'  without owner (self-terminals): {len(no_owner)}')

    sub2.to_csv(f'ultimate_owners_{mode}_noself.csv', index=False)
    print(f'  Saved: ultimate_owners_{mode}_noself.csv')


Total rows: 649
Total manufacturers: 196

Distribution by mode:
            manufacturer  ultimateOwner
mode                                   
all_links            196            108
controlish           196            102

== all_links ==
  total manufacturers:          196
  with at least 1 non-self owner:  176
  without owner (self-terminals): 20
  Saved: ultimate_owners_all_links_noself.csv

== controlish ==
  total manufacturers:          196
  with at least 1 non-self owner:  168
  without owner (self-terminals): 28
  Saved: ultimate_owners_controlish_noself.csv


In [6]:
# Noise Filter

ULT_IN   = 'ultimate_owners.csv'
NODES_IN = 'ownership_nodes.csv'
ULT_OUT  = 'ultimate_owners_clean.csv'

ult   = pd.read_csv(ULT_IN)
nodes = pd.read_csv(NODES_IN)

label_by  = nodes.groupby('entity')['entityLabel'].first().fillna('').to_dict()
p31uri_by = nodes.groupby('entity')['p31'].apply(
    lambda s: {str(x).strip() for x in s.dropna() if str(x).strip()}
).to_dict()

def get_label(e): return str(label_by.get(str(e), '')).strip().lower()
def get_p31uri(e): return p31uri_by.get(str(e), set())

# P31 URI Wikimedia: meta pages, not real entities (100% sure)
WIKIMEDIA_P31 = {
    'http://www.wikidata.org/entity/Q4167836',   # Wikimedia category
    'http://www.wikidata.org/entity/Q13406463',  # Wikimedia list article
    'http://www.wikidata.org/entity/Q17362920',  # Wikimedia disambiguation page
}

# Label exact to filter (lowercase)
# Derives from observed results: only manually verified entities
LABEL_EXACT = {
    # ── stock indices (not companies)
    's&p 500', 's&p 100',
    'dow jones global titans 50',
    'ftse 100 index',
    'dax', 'cdax', 'divdax', 'dax indexes',
    'prime standard', 'nikkei 225', 'nasdaq-100',
    'rts index', 'micex index', 'micex 10',
    # ── stock exchanges / financial markets (not industrial owners)
    'euronext',
    'cme group',
    'regulated market',
    'japan exchange group',
    # ── generic wikidata concepts (not real entities)
    'investor',
    'institutional investor',
    'industrial sector',
    'history of poland',
    'consortium',
    # ── abstract financial concepts (not owner entities)
    'employee stock ownership',
    'public float',
    # ── generic financial funds (not industrial owners)
    'delaware management business trust',
    # ── astronomical / cosmic entities (emerged for anomalous P361 chains)
    'solar system',
    'earth-moon system',
    "earth's surface",
    'ostfeste',
    'local group',
    'local bubble',
    'milky way subgroup',
    'inner and outer solar system',
    # ── abstract geographical regions (not owner entities)
    'eastern europe',
    'menasa',
    'geographic envelope',
    # ── various anomalous entries in the dataset
    'economy of russia', 'industry in russia', 'near and middle east',
    'ford family tree', 'employment contract', 'two sessions',
    # ── historical entities dissolved (cannot be current owners)
    'reichswerke ag für waffen-und maschinenbau hermann göring',
    # ── historical people (cannot be current owners)
    'william armstrong, 1st baron armstrong',
}

# Label that starts with certain noise prefix
LABEL_STARTSWITH = [
    'list of',      
    'system of',
]

# Label that contains certain noise substring
LABEL_CONTAINS = [
    'competition between',    # "competition between Airbus and Boeing"
]

# QID not resolved: label == raw QID 
_unresolved_qid_re = re.compile(r'^q\d+$')


BLANK_NODE_PREFIX = 'http://www.wikidata.org/.well-known/genid/'

def is_noise(uri):
    uri_str = str(uri)
    if uri_str.startswith(BLANK_NODE_PREFIX):
        return True
    if get_p31uri(uri) & WIKIMEDIA_P31:
        return True
    lab = get_label(uri)
    if lab in LABEL_EXACT:
        return True
    if any(lab.startswith(h) for h in LABEL_STARTSWITH):
        return True
    if any(h in lab for h in LABEL_CONTAINS):
        return True
    if _unresolved_qid_re.match(lab):
        return True
    return False

ult['is_noise'] = ult['ultimateOwner'].apply(is_noise)
removed = ult[ult['is_noise']]
kept    = ult[~ult['is_noise']].drop(columns=['is_noise'])

kept.to_csv(ULT_OUT, index=False)

print(f'Righe originali: {len(ult)}')
print(f'Righe rumore:    {len(removed)}')
print(f'Righe pulite:    {len(kept)}')
print(f'Saved: {ULT_OUT}')

print('\nEntità rimosse come rumore:')
print(removed.groupby(['ultimateOwner','ultimateOwnerLabel']).size()
             .sort_values(ascending=False)
             .rename('n_righe')
             .reset_index()
             .to_string(index=False))

print('\nVerifica: owner rimasti (sample) — assicurarsi che non ci siano falsi negativi ovvi:')
kept_owners = kept.groupby(['ultimateOwner','ultimateOwnerLabel']).size()\
                  .sort_values(ascending=False).reset_index()
print(kept_owners.tail(20)[['ultimateOwnerLabel']].to_string(index=False))


Righe originali: 649
Righe rumore:    44
Righe pulite:    605
Saved: ultimate_owners_clean.csv

Entità rimosse come rumore:
                                                             ultimateOwner                                                         ultimateOwnerLabel  n_righe
                                   http://www.wikidata.org/entity/Q1665188                                                     institutional investor       14
                                    http://www.wikidata.org/entity/Q557880                                                                   investor       14
                                 http://www.wikidata.org/entity/Q107174045                  Reichswerke AG für Waffen-und Maschinenbau Hermann Göring        4
http://www.wikidata.org/.well-known/genid/fa394b152e4bf3a70b022b174ef3033f http://www.wikidata.org/.well-known/genid/fa394b152e4bf3a70b022b174ef3033f        2
                                    http://www.wikidata.org/entity/Q335115       

In [7]:
# Cross-conflict analysis

ult_clean = pd.read_csv('ultimate_owners_clean.csv')
nodes_fix = pd.read_csv('ownership_nodes.csv')


KNOWN_QID_LABELS = {
    'Q785':  'Jersey',       # British Crown dependency
}

_qid_raw_re = re.compile(r'^Q\d+$')
_uri_to_label = (
    nodes_fix.drop_duplicates(subset='entity')
    .set_index('entity')['entityLabel']
    .to_dict()
)

def _resolve_country(val):
    s = str(val).strip()
    if not _qid_raw_re.match(s):
        return val  
    if s in KNOWN_QID_LABELS:
        return KNOWN_QID_LABELS[s]
    uri = f'http://www.wikidata.org/entity/{s}'
    lab = _uri_to_label.get(uri, '')
    return lab if lab and not _qid_raw_re.match(lab) else val

ult_clean['ultimateOwnerCountry'] = ult_clean['ultimateOwnerCountry'].apply(_resolve_country)


still_qid = ult_clean['ultimateOwnerCountry'].str.match(r'^Q\d+$', na=False).sum()
print(f'Raw QIDs still in ultimateOwnerCountry after fix: {still_qid}')
if still_qid > 0:
    print('Still not resolved (add to KNOWN_QID_LABELS):')
    print(ult_clean[ult_clean['ultimateOwnerCountry'].str.match(r'^Q\d+$', na=False)]
          [['ultimateOwnerLabel','ultimateOwnerCountry']].drop_duplicates().to_string(index=False))

ult_clean.to_csv('ultimate_owners_clean.csv', index=False)

print(f'\nRows after noise filter: {len(ult_clean)}')
print(f'Manufacturer: {ult_clean["manufacturer"].nunique()}')
print(f'Ultimate owners unici: {ult_clean["ultimateOwner"].nunique()}')

# top 15 ultimate owners per mode (on clean dataset)
for mode in ult_clean['mode'].unique():
    sub  = ult_clean[ult_clean['mode'] == mode].copy()
    sub2 = sub[sub['manufacturer'] != sub['ultimateOwner']].copy()

    print(f'\n== {mode} — top 15 ultimate owners (after noise filter) ==')
    top = (
        sub2.groupby(['ultimateOwnerLabel','ultimateOwnerCountry'])
        .agg(n_manufacturer=('manufacturer','nunique'))
        .sort_values('n_manufacturer', ascending=False)
        .head(15)
    )
    print(top.to_string())


print('\n── Owner cross-conflitto (controlish, post noise filter):')
ctrl = ult_clean[ult_clean['mode'] == 'controlish'].copy()
ctrl = ctrl[ctrl['manufacturer'] != ctrl['ultimateOwner']].copy()

ctrl['conflict_list'] = ctrl['conflict'].str.split('|')
ctrl_exp = ctrl.explode('conflict_list')

cross = (
    ctrl_exp.groupby(['ultimateOwnerLabel','ultimateOwnerCountry'])
    ['conflict_list'].nunique()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={'conflict_list': 'n_conflitti'})
)
cross = cross[cross['n_conflitti'] >= 2].reset_index(drop=True)
print(cross.to_string(index=False))
cross.to_csv('ultimate_owners_cross_conflict.csv', index=False)
print(f'\nSaved: ultimate_owners_cross_conflict.csv ({len(cross)} owner cross-conflitto)')

# heatmap owner x conflitto 
print('\n── Matrice owner × conflitto (controlish, post noise filter):')
pivot = (
    ctrl_exp[ctrl_exp['ultimateOwnerLabel'].isin(cross['ultimateOwnerLabel'])]
    .groupby(['ultimateOwnerLabel','conflict_list'])
    .agg(n_manufacturer=('manufacturer','nunique'))
    .reset_index()
    .pivot(index='ultimateOwnerLabel', columns='conflict_list', values='n_manufacturer')
    .fillna(0)
    .astype(int)
)
print(pivot.to_string())
pivot.to_csv('owner_conflict_matrix.csv')
print(f'\nSaved: owner_conflict_matrix.csv — input principale per M4')


Raw QIDs still in ultimateOwnerCountry after fix: 0

Rows after noise filter: 605
Manufacturer: 194
Ultimate owners unici: 145

== all_links — top 15 ultimate owners (after noise filter) ==
                                                       n_manufacturer
ultimateOwnerLabel         ultimateOwnerCountry                      
Russia                     Russia                                  43
BlackRock                  United States                           24
United States              United States                           23
The Vanguard Group         United States                           22
France                     France                                  13
State Street Corporation   United States                           12
Capital Group Companies    United States                            9
Bank of America            United States                            9
Germany                    Germany                                  9
Ukraine                    Ukraine      